# video-notes：在 Colab T4 GPU 上生成字幕（SRT）

本地没有 NVIDIA 显卡时，长视频的语音识别很慢。这个笔记本在 Colab 的 T4 GPU 上用 WhisperX 生成**带精确时间轴的 SRT**，下载后放到视频旁边，本地 `video-notes` 会自动使用。

步骤：
1. 菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**（Runtime → Change runtime type → T4 GPU）。
2. 从上到下依次运行每个单元格。
3. 把得到的 `<视频名>.srt` 放到本地视频旁边（文件名与视频相同，只是扩展名为 .srt）。

提示：只需要音频。可以先在本地运行
`ffmpeg -i "课程.mp4" -vn -ac 1 -c:a aac -b:a 64k "课程.m4a"`
把 800 MB 的视频变成约 75 MB 的音频再上传，文件名主干保持与视频相同。

In [ ]:
# 1. 确认已分配 GPU（必须是 T4 或其他 NVIDIA GPU）
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), '没有 GPU：请在“更改运行时类型”中选择 T4 GPU 后重新运行'

In [ ]:
# 2. 安装 WhisperX（约 2-4 分钟）
!pip -q install whisperx

In [ ]:
# 3. 参数
SOURCE = 'upload'          # 'upload' = 从本机上传；'drive' = 读取 Google Drive 中的文件
DRIVE_FILE = '/content/drive/MyDrive/videos/课程.m4a'   # SOURCE='drive' 时使用；SRT 会写在同一文件夹
MODEL = 'large-v3'         # 精度最高；更快可用 'medium'
LANGUAGE = None            # None = 自动识别；也可指定 'en'、'zh' 等
INITIAL_PROMPT = ''        # 可选：领域术语提示，例如 'Technical terms include BGP, OSPF, VXLAN.'
BATCH_SIZE = 16            # T4 上 16 通常可行；显存不足时改为 8

In [ ]:
# 4. 取得音频或视频文件
from pathlib import Path
if SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    media = Path(DRIVE_FILE)
else:
    from google.colab import files
    uploaded = files.upload()
    media = Path('/content') / next(iter(uploaded))
assert media.is_file(), media
print('输入文件：', media)

In [ ]:
# 5. 转写 + 词级对齐（2 小时音频在 T4 上通常十几分钟，视模型与内容而定）
import time, whisperx
started = time.time()
model = whisperx.load_model(MODEL, 'cuda', compute_type='float16', language=LANGUAGE,
                            asr_options={'initial_prompt': INITIAL_PROMPT or None})
audio = whisperx.load_audio(str(media))
result = model.transcribe(audio, batch_size=BATCH_SIZE)
language = result['language']
align_model, metadata = whisperx.load_align_model(language_code=language, device='cuda')
result = whisperx.align(result['segments'], align_model, metadata, audio, 'cuda', return_char_alignments=False)
print(f'语言 {language}，{len(result["segments"])} 段，用时 {time.time() - started:.0f} 秒')

In [ ]:
# 6. 写出 SRT（与输入文件同名）并下载
def stamp(seconds):
    ms = int(round(seconds * 1000))
    return f'{ms // 3600000:02}:{ms // 60000 % 60:02}:{ms // 1000 % 60:02},{ms % 1000:03}'

segments = [s for s in result['segments'] if s.get('text', '').strip() and 'start' in s and 'end' in s]
srt = media.with_suffix('.srt') if SOURCE == 'drive' else Path('/content') / (media.stem + '.srt')
srt.write_text(''.join(f"{i}
{stamp(s['start'])} --> {stamp(s['end'])}
{s['text'].strip()}

"
                       for i, s in enumerate(segments, 1)), encoding='utf-8')
print('已生成', srt, f'（{len(segments)} 条字幕）')
if SOURCE != 'drive':
    from google.colab import files
    files.download(str(srt))